# Day 1 - Topic 6: Comprehensions and Generator Expressions

> Lead-Level Data Science Interview Prep Series

## 1. Introduction

- **Comprehensions** are a compact, one-line way to build a list, dict, or set from an existing iterable, replacing a multi-line loop
- **Generator expressions** look similar to list comprehensions but produce values **one at a time**, lazily, instead of building the whole collection in memory at once
- Why needed?
  - Cleaner, more Pythonic code - considered a sign of Python fluency in interviews
  - Generators solve the real problem of processing huge datasets without running out of memory
- Where used?
  - Quick data transformations (filtering, mapping) everywhere in Data Science code
  - Generators are used when streaming large files/datasets row by row instead of loading everything into RAM

## 2. Real-Life Analogy

- A **list comprehension** is like a factory assembly line that builds ALL the products first, then hands you the entire finished batch in one box (all in memory at once)
- A **generator expression** is like a chef who cooks ONE dish at a time, only when you ask for the next one - nothing is prepared in advance, saving effort/resources
- If you only need to look at each item once and don't need to store all of them, the generator (chef) approach is far more efficient than the factory (batch) approach

## 3. Explanation

- **List comprehension**: `[expression for item in iterable if condition]` - builds and stores the full list in memory immediately
- **Dict comprehension**: `{key_expr: value_expr for item in iterable if condition}` - builds a full dictionary
- **Set comprehension**: `{expression for item in iterable if condition}` - builds a full set (auto-deduplicates)
- **Generator expression**: `(expression for item in iterable if condition)` - looks like a list comprehension but with `()` instead of `[]`; does NOT build anything upfront - it creates a generator object that yields values one at a time, only when asked

> **Trick to remember:** The brackets tell you the type - `[]` = List, `{}` = Dict/Set, `()` = Generator. Same pattern, different container, different memory behavior.

## 4. Syntax

```python
# List comprehension
[expression for item in iterable]
[expression for item in iterable if condition]

# Dict comprehension
{key: value for item in iterable}

# Set comprehension
{expression for item in iterable}

# Generator expression
(expression for item in iterable)
```

- `expression` = what to compute/store for each item
- `for item in iterable` = the loop part
- `if condition` = optional filter - only items passing this are included

In [ ]:
# Quick syntax demo of all four
nums = [1, 2, 3, 4, 5]

list_comp = [n * n for n in nums]
dict_comp = {n: n * n for n in nums}
set_comp = {n % 2 for n in nums}     # only unique remainders: {0, 1}
gen_exp = (n * n for n in nums)

print(list_comp)
print(dict_comp)
print(set_comp)
print(gen_exp)          # shows a generator object, not the values
print(list(gen_exp))    # convert to list to see values (can only do this once!)


## 5. Examples

### Basic Example

In [ ]:
# Basic: list comprehension vs traditional loop (same result)
squares_loop = []
for n in range(1, 6):
    squares_loop.append(n * n)

squares_comp = [n * n for n in range(1, 6)]

print(squares_loop)
print(squares_comp)


### Intermediate Example

In [ ]:
# Intermediate: filtering with a condition, and building a dict comprehension
scores = [45, 88, 62, 39, 91, 55]

passing_scores = [s for s in scores if s >= 50]
print("Passing scores:", passing_scores)

grade_labels = {s: ("Pass" if s >= 50 else "Fail") for s in scores}
print("Grade labels:", grade_labels)


- `[s for s in scores if s >= 50]` is a filtered list comprehension - only keeps items where the condition is `True`
- The dict comprehension uses an inline `if/else` **expression** (not the `if` filter) to decide the value for every key - note the different role `if` plays here versus the filtering example above

### Real-World Example

In [ ]:
# Real-world: cleaning + transforming raw data in one line, and a memory-efficient generator
# This is the Pythonic replacement for the manual cleaning loop from the earlier Loops topic

raw_ages = [25, -5, 34, 150, 41, "unknown", 29]

# List comprehension: clean valid ages in one line
clean_ages = [a for a in raw_ages if isinstance(a, int) and 0 <= a <= 120]
print("Clean ages:", clean_ages)

# Generator expression: sum of squares WITHOUT building a full list in memory
# Useful when raw_ages could be millions of rows from a huge file
sum_of_squares = sum(a * a for a in clean_ages)
print("Sum of squares:", sum_of_squares)


- The list comprehension replaces the earlier multi-line loop-with-`continue` pattern from the Loops topic - same logic, one line
- `sum(a * a for a in clean_ages)` passes a **generator expression directly into `sum()`** without ever creating an intermediate list - this is the single most common real-world use of generator expressions
- Note: no extra `()` needed when a generator expression is the only argument to a function like `sum()`, `max()`, `any()`

## 6. Internal Working

- List/dict/set comprehensions run the entire loop immediately and store every result in memory - functionally similar to a `for` loop with `.append()`, but faster because Python optimizes the comprehension bytecode
- A generator expression does NOT run the loop immediately - it creates a **generator object** that implements the iterator protocol (`__iter__`/`__next__`)
- Each time you call `next()` on a generator (or a `for` loop pulls from it), it runs the code just far enough to produce ONE value, then **pauses** - resuming exactly where it left off on the next call
- A generator can only be consumed **once** - after it's exhausted, iterating again gives nothing

> **Trick to remember:** "Comprehension = eager (does everything now). Generator = lazy (does it only when asked, one at a time)." This eager-vs-lazy distinction is the #1 interview question on this topic.

In [ ]:
gen = (n * n for n in range(3))
print(list(gen))   # [0, 1, 4]
print(list(gen))   # [] - already exhausted, generator can only be consumed once!


## 7. Time and Space Complexity

- **List/Dict/Set comprehension**: O(n) time (one pass through the iterable), O(n) space (stores all n results in memory)
- **Generator expression**: O(n) time overall (still processes every item eventually), but **O(1) space** - it never stores more than the current item, regardless of how large the input is

> **Interview line:** For a dataset with 10 million rows where I just need to sum/filter once, I'd use a generator expression - O(1) space instead of O(n) - because I never need all values in memory simultaneously.

## 8. Common Mistakes

- Trying to reuse a generator after it's been fully consumed - it will silently return nothing (no error), which can cause confusing bugs
- Using a list comprehension when a generator would be enough (e.g. inside `sum()`, `max()`, `any()`) - wastes memory unnecessarily on large data
- Confusing set comprehension `{x for x in ...}` with dict comprehension `{k: v for x in ...}` - the colon `:` is what makes it a dict
- Writing overly complex, deeply nested comprehensions that hurt readability - if it needs a comment to explain, a normal loop is often better
- Forgetting that comprehensions create a **new** variable scope in Python 3 - the loop variable doesn't leak into the surrounding scope (unlike a plain `for` loop)

In [ ]:
# Common mistake: assuming the loop variable leaks out (it does NOT in comprehensions, Python 3)
squares = [n * n for n in range(5)]
try:
    print(n)   # NameError - n only existed inside the comprehension
except NameError as e:
    print("Error:", e)


## 9. Best Practices

- Use comprehensions for simple, readable transformations - avoid nesting more than 2 levels deep
- Use a **generator expression** instead of a list comprehension when you only need to iterate once and don't need to store all values (saves memory)
- Prefer a normal `for` loop over a comprehension if the logic needs multiple statements or is hard to read as one line
- Use dict comprehensions to quickly build lookup tables (e.g. `{id: name for id, name in records}`) instead of manual loops
- Name comprehension results clearly (`clean_ages`, not `x`) even though the loop variable itself can be short

## 10. Interview Questions

**Beginner**
- Q: What is a list comprehension?
  A: A concise, one-line way to create a list by applying an expression to each item in an iterable, optionally filtered by a condition - a compact replacement for a `for` loop with `.append()`.
- Q: How do you write a comprehension that only keeps even numbers from a list?
  A: `[n for n in numbers if n % 2 == 0]`

**Intermediate**
- Q: What is the key difference between a list comprehension and a generator expression?
  A: A list comprehension builds and stores the entire result in memory immediately (eager evaluation); a generator expression produces values one at a time, on demand, without storing them all (lazy evaluation) - saving memory.
- Q: Can you reuse a generator after iterating through it once?
  A: No - once exhausted, a generator is empty. You'd need to recreate it to iterate again.

**Advanced**
- Q: When would you specifically choose a generator expression over a list comprehension in a Data Science pipeline?
  A: When processing very large datasets where you only need to pass through the data once (e.g. summing, filtering row by row from a large file) and don't need random access or to store all results - this keeps memory usage O(1) instead of O(n).
- Q: What's the difference in variable scoping between a comprehension and a traditional `for` loop in Python 3?
  A: Comprehensions have their own local scope in Python 3 - the loop variable does not leak into the enclosing scope after the comprehension finishes. A regular `for` loop's variable does persist in the enclosing scope after the loop ends.

## 11. Practice Problems

**Easy**
1. Write a list comprehension to square all numbers from 1 to 10.
2. Write a set comprehension to get all unique first letters from a list of names.

**Medium**
3. Given a list of words, write a dict comprehension mapping each word to its length.
4. Rewrite the "real-world" cleaning example from the Loops topic (filtering valid ages) using a single list comprehension.

**Hard**
5. Write a generator expression that computes the sum of squares of only the even numbers from 1 to 1,000,000, and explain (in a comment) why this approach uses less memory than first building a full list of squares.

## 12. Revision Summary

- List comprehension: `[expr for item in iterable if cond]` - eager, stores everything, O(n) space
- Dict comprehension: `{key: value for item in iterable}` - builds a lookup table in one line
- Set comprehension: `{expr for item in iterable}` - auto-deduplicates
- Generator expression: `(expr for item in iterable)` - lazy, O(1) space, values produced one at a time
- A generator can only be consumed once - it's exhausted after first full iteration
- Comprehensions have their own scope - loop variable does not leak out (Python 3)
- Use generators for large data / single-pass processing; use comprehensions for smaller, reusable results

> **Day 1 complete.** Next: **Day 2 - Functions & OOP**